In [1]:
"""
POS Tagging Examples in Python
Author: Phd Abdelouaheb

This file shows several ways to do POS tagging:
1) NLTK built-in tagger (simple & educational)
2) spaCy pipeline (production-friendly)
3) NLTK backoff (RegexTagger + UnigramTagger)
4) Mapping PTB tags to Universal POS
5) Counting and simple pattern mining
NOTE: Installation commands are commented out. Uncomment in your own environment if needed.
"""

# --- Setup (uncomment as needed) ---
# pip install nltk spacy
# python -m spacy download en_core_web_sm
# python -m nltk.downloader punkt averaged_perceptron_tagger averaged_perceptron_tagger_eng universal_tagset

import re
from collections import Counter

# ============== 1) NLTK: basic tagging ==============
try:
    import nltk
    from nltk import word_tokenize, pos_tag
    nltk.download("punkt", quiet=True)
    # For older/newer NLTK versions, sometimes need:
    # nltk.download("averaged_perceptron_tagger", quiet=True)
    # nltk.download("averaged_perceptron_tagger_eng", quiet=True)
    # nltk.download("universal_tagset", quiet=True)
except Exception as e:
    nltk = None
    print("[WARN] NLTK not available:", e)

text = "The quick brown fox jumps over the lazy dog near the river bank in Paris."

if nltk is not None:
    tokens = word_tokenize(text)
    ptb_tags = pos_tag(tokens)  # Penn Treebank tags by default
    print("\n[NLTK] PTB tags:")
    print(ptb_tags)

    # Optionally map to Universal tagset (if available)
    try:
        upos_tags = nltk.pos_tag(tokens, tagset="universal")
        print("\n[NLTK] Universal POS tags:")
        print(upos_tags)
    except Exception as e:
        print("[WARN] Could not map to Universal tagset:", e)

# ============== 2) spaCy: pipeline tagging ==============
try:
    import spacy
    # Load small English model (must be downloaded ahead of time)
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    print("\n[spaCy] Tokens with POS, TAG (fine-grained), and DEP (dependency):")
    for token in doc:
        print(f"{token.text:10s} POS={token.pos_:6s} TAG={token.tag_:5s} DEP={token.dep_:8s}")
except Exception as e:
    print("\n[spaCy] Not available or model missing:", e)

# ============== 3) NLTK: Backoff taggers ==============
if nltk is not None:
    from nltk.tag import RegexpTagger, UnigramTagger, DefaultTagger

    # DefaultTagger: fallback to NOUN (NN)
    default_tagger = DefaultTagger("NN")

    # Simple regex rules (very small illustrative set)
    regex_patterns = [
        (r".*ing$", "VBG"),   # gerunds
        (r".*ed$", "VBD"),    # past tense verbs
        (r".*ly$", "RB"),     # adverbs
        (r".*ous$", "JJ"),    # adjectives
        (r"^[0-9]+(\.[0-9]+)?$", "CD"),  # numbers
        (r".*", "NN"),        # nouns (fallback)
    ]
    regex_tagger = RegexpTagger(regex_patterns, backoff=default_tagger)

    # Train a tiny UnigramTagger on a toy corpus (for demo)
    train_sents = [
        [("The","DT"), ("cat","NN"), ("sat","VBD")],
        [("The","DT"), ("dog","NN"), ("sat","VBD")],
        [("A","DT"), ("quick","JJ"), ("fox","NN")],
    ]
    unigram_tagger = UnigramTagger(train=train_sents, backoff=regex_tagger)

    tokens2 = nltk.word_tokenize("A curious cat swiftly jumped")
    print("\n[NLTK] Backoff tagging (Unigram -> Regex -> Default):")
    print(unigram_tagger.tag(tokens2))

# ============== 4) Map PTB -> Universal manually (fallback) ==============
# Minimal demo map for a few tags; extend as needed.
PTB_TO_UPOS = {
    "NN": "NOUN", "NNS": "NOUN", "NNP": "PROPN", "NNPS": "PROPN",
    "VB": "VERB", "VBD": "VERB", "VBG": "VERB", "VBN": "VERB", "VBP": "VERB", "VBZ": "VERB",
    "JJ": "ADJ", "JJR": "ADJ", "JJS": "ADJ",
    "RB": "ADV", "RBR": "ADV", "RBS": "ADV",
    "IN": "ADP", "DT": "DET", "PRP": "PRON", "PRP$": "DET",
    "CC": "CCONJ", "CD": "NUM", "UH": "INTJ", "TO": "PART", "MD": "AUX",
    "POS": "PART", "EX": "PRON", "FW": "X", "SYM": "SYM", "LS": "X", "PDT": "DET",
    ".": "PUNCT", ",": "PUNCT", ":": "PUNCT", "''": "PUNCT", "``": "PUNCT",
}

def ptb_to_upos(ptb_tag: str) -> str:
    return PTB_TO_UPOS.get(ptb_tag, "X")

sample_ptb = [("Paris","NNP"), ("is","VBZ"), ("beautiful","JJ"), (".",".")]
mapped = [(w, ptb_to_upos(t)) for w, t in sample_ptb]
print("\n[Manual] PTB → Universal mapping (demo):")
print(mapped)

# ============== 5) Counting & simple patterns ==============
def count_upos(tagged_tokens, is_upos=True):
    """
    Count tag frequencies.
    tagged_tokens: list of (word, tag) pairs.
    is_upos: if False (PTB), we map using PTB_TO_UPOS.
    """
    if not is_upos:
        tagged_tokens = [(w, ptb_to_upos(t)) for w, t in tagged_tokens]
    return Counter(tag for _, tag in tagged_tokens)

if nltk is not None:
    # If we have UPOS from NLTK
    try:
        upos = nltk.pos_tag(nltk.word_tokenize(text), tagset="universal")
        print("\n[Counts] UPOS distribution:")
        print(count_upos(upos, is_upos=True))
    except Exception:
        ptb = nltk.pos_tag(nltk.word_tokenize(text))
        print("\n[Counts] UPOS distribution (mapped from PTB):")
        print(count_upos(ptb, is_upos=False))

# Chunk pattern example: extract ADJ+NOUN phrases from spaCy if available
try:
    import spacy
    nlp = spacy.load("en_core_web_sm")
    doc = nlp("We need compact wireless keyboards and fast external SSDs for travel.")
    phrases = []
    for i in range(len(doc)-1):
        if doc[i].pos_ == "ADJ" and doc[i+1].pos_ in {"NOUN", "PROPN"}:
            phrases.append(doc[i].text + " " + doc[i+1].text)
    print("\n[spaCy] ADJ+NOUN phrases:", phrases)
except Exception:
    pass

print("\nDone.")



[NLTK] PTB tags:
[('The', 'DT'), ('quick', 'JJ'), ('brown', 'NN'), ('fox', 'NN'), ('jumps', 'VBZ'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN'), ('near', 'IN'), ('the', 'DT'), ('river', 'NN'), ('bank', 'NN'), ('in', 'IN'), ('Paris', 'NNP'), ('.', '.')]
[WARN] Could not map to Universal tagset: 
**********************************************************************
  Resource universal_tagset not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('universal_tagset')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load taggers/universal_tagset/en-ptb.map

  Searched in:
    - 'C:\\Users\\ASUS TUF/nltk_data'
    - 'c:\\Program Files\\Python312\\nltk_data'
    - 'c:\\Program Files\\Python312\\share\\nltk_data'
    - 'c:\\Program Files\\Python312\\lib\\nltk_data'
    - 'C:\\Users\\ASUS TUF\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
    - ''